In [1]:
import numpy as np
import pandas as pd
import os
import sys
import subprocess
from pathlib import Path
for dirname, _, filenames in os.walk('/kaggle/input'):
    for filename in filenames:
        print(os.path.join(dirname, filename))

In [2]:
import matplotlib.pyplot as plt
import seaborn as sns
import re
import warnings
warnings.filterwarnings('ignore')

import nltk
nltk.download('stopwords', quiet=True)
nltk.download('wordnet', quiet=True)
nltk.download('punkt', quiet=True)
nltk.download('punkt_tab', quiet=True)
from nltk.corpus import stopwords
from nltk.stem import WordNetLemmatizer

from scipy.stats import loguniform, randint

from sklearn.model_selection import (
    train_test_split, cross_val_score, StratifiedKFold,
    RandomizedSearchCV, learning_curve
)
from sklearn.calibration import CalibratedClassifierCV
from sklearn.base import clone
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.linear_model import LogisticRegression
from sklearn.svm import LinearSVC
from sklearn.ensemble import RandomForestClassifier
from sklearn.naive_bayes import MultinomialNB
from sklearn.metrics import (
    classification_report, confusion_matrix, accuracy_score, f1_score,
    precision_recall_fscore_support, ConfusionMatrixDisplay
)

print("Done.")

Done.


In [3]:
from pathlib import Path
print("cwd =", Path.cwd())
print("target =", Path("data").resolve())

cwd = /workspace
target = /workspace/data


In [4]:
URL = "/kaggle/input/datasets/mrmorj/hate-speech-and-offensive-language-dataset/labeled_data.csv"

if not os.path.exists(URL):
    URL = Path("data/labeled_data.csv")
    print("target =", URL.resolve())
    print("exists =", URL.exists())
    if not URL.exists():
        URL.parent.mkdir(parents=True, exist_ok=True)
        print("exists =", URL.parent.exists())
        result = subprocess.run(
            ["bash","-c", "curl -L -o data/hate-speech-and-offensive-language-dataset.zip \
            https://www.kaggle.com/api/v1/datasets/download/mrmorj/hate-speech-and-offensive-language-dataset"],
            capture_output=True,
            text=True,
            check=False
        )
        print(result.stdout)
        result = subprocess.run(
            ["bash","-c", "ls -la data"],
            capture_output=True,
            text=True,
            check=False
        )
        print(result.stdout)
        result = subprocess.run(
            ["bash","-c", "unzip data/hate-speech-and-offensive-language-dataset.zip -d data"],
            capture_output=True,
            text=True,
            check=False
        )
        print(result.stdout)
        result = subprocess.run(
            ["bash","-c", "ls -la data"],
            capture_output=True,
            text=True,
            check=False
        )
        print(result.stdout)
df = pd.read_csv(URL)

print(f"Shape: {df.shape}")
df.head()

target = /workspace/data/labeled_data.csv
exists = True
Shape: (24783, 7)


,Unnamed: 0,count,hate_speech,offensive_language,neither,class,tweet
0,0,3,0,0,3,2,!!! RT @mayasolovely: As a woman you shouldn't...
1,1,3,0,3,0,1,!!!!! RT @mleew17: boy dats cold...tyga dwn ba...
2,2,3,0,3,0,1,!!!!!!! RT @UrKindOfBrand Dawg!!!! RT @80sbaby...
3,3,3,0,2,1,1,!!!!!!!!! RT @C_G_Anderson: @viva_based she lo...
4,4,6,0,6,0,1,!!!!!!!!!!!!! RT @ShenikaRoberts: The shit you...


In [5]:
print(df.info())
print("\n" + "="*50)
print(df.describe())
print("\n" + "="*50)
print("\nClass distribution:")
print(df['class'].value_counts())
print("\nNull values:")
print(df.isnull().sum())

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 24783 entries, 0 to 24782
Data columns (total 7 columns):
 #   Column              Non-Null Count  Dtype 
---  ------              --------------  ----- 
 0   Unnamed: 0          24783 non-null  int64 
 1   count               24783 non-null  int64 
 2   hate_speech         24783 non-null  int64 
 3   offensive_language  24783 non-null  int64 
 4   neither             24783 non-null  int64 
 5   class               24783 non-null  int64 
 6   tweet               24783 non-null  object
dtypes: int64(6), object(1)
memory usage: 1.3+ MB
None

         Unnamed: 0         count   hate_speech  offensive_language  \
count  24783.000000  24783.000000  24783.000000        24783.000000   
mean   12681.192027      3.243473      0.280515            2.413711   
std     7299.553863      0.883060      0.631851            1.399459   
min        0.000000      3.000000      0.000000            0.000000   
25%     6372.500000      3.000000      0.000000  

In [6]:
label_map = {0: 'Hate Speech', 1: 'Offensive Language', 2: 'Neither'}
fig, axes = plt.subplots(1, 2, figsize=(14, 5))

counts = df['class'].value_counts().sort_index()
colors = ['#e74c3c', '#f39c12', '#2ecc71']
bars = axes[0].bar([label_map[i] for i in counts.index], counts.values, color=colors, edgecolor='black')
for bar, val in zip(bars, counts.values):
    axes[0].text(bar.get_x() + bar.get_width()/2, bar.get_height() + 200,
                 f'{val:,}', ha='center', fontsize=12, fontweight='bold')
axes[0].set_title('Class Distribution', fontsize=14, fontweight='bold')
axes[0].set_ylabel('Count')

axes[1].pie(counts.values, labels=[label_map[i] for i in counts.index],
            autopct='%1.1f%%', colors=colors, startangle=90,
            textprops={'fontsize': 12}, wedgeprops={'edgecolor': 'black'})
axes[1].set_title('Class Proportions', fontsize=14, fontweight='bold')

plt.tight_layout()
plt.show()

In [7]:
df['tweet_length'] = df['tweet'].str.len()
df['word_count'] = df['tweet'].str.split().str.len()

fig, axes = plt.subplots(1, 2, figsize=(14, 5))
for cls, color, label in [(0, '#e74c3c', 'Hate Speech'),
                           (1, '#f39c12', 'Offensive'),
                           (2, '#2ecc71', 'Neither')]:
    subset = df[df['class'] == cls]
    axes[0].hist(subset['tweet_length'], bins=50, alpha=0.6, label=label, color=color)
    axes[1].hist(subset['word_count'], bins=30, alpha=0.6, label=label, color=color)

axes[0].set_title('Tweet Length Distribution', fontweight='bold')
axes[0].set_xlabel('Character count')
axes[0].legend()
axes[1].set_title('Word Count Distribution', fontweight='bold')
axes[1].set_xlabel('Word count')
axes[1].legend()
plt.tight_layout()
plt.show()

In [8]:
# Correlations between numeric variables
num_cols = [c for c in ['count', 'hate_speech', 'offensive_language', 'neither', 'tweet_length', 'word_count'] if c in df.columns]
corr_df = df[num_cols + ['class']].copy()

plt.figure(figsize=(9, 6))
sns.heatmap(corr_df.corr(numeric_only=True), annot=True, fmt='.2f', cmap='coolwarm', square=True)
plt.title('Correlation Heatmap (Numeric Features + Class)', fontweight='bold')
plt.tight_layout()
plt.show()

In [9]:
lemmatizer = WordNetLemmatizer()
stop_words = set(stopwords.words('english'))

def preprocess_text(text):
    text = str(text).lower()
    text = re.sub(r'http\S+|www\S+|https\S+', '', text)
    text = re.sub(r'@\w+', '', text)
    text = re.sub(r'#(\w+)', r'\1', text)
    text = re.sub(r'rt\s+', '', text)
    text = re.sub(r'&amp;|&lt;|&gt;', '', text)
    text = re.sub(r'[^a-z\s]', '', text)
    text = re.sub(r'\s+', ' ', text).strip()
    tokens = text.split()
    tokens = [lemmatizer.lemmatize(w) for w in tokens if w not in stop_words and len(w) > 2]
    return ' '.join(tokens)

df['clean_tweet'] = df['tweet'].apply(preprocess_text)

df = df[df['clean_tweet'].str.len() > 0].reset_index(drop=True)

print(f"Dataset after cleaning: {df.shape[0]} rows")
print("\nBefore - After examples:")
for i in [0, 100, 500]:
    print(f"  Original:  {df['tweet'].iloc[i][:80]}...")
    print(f"  Cleaned:   {df['clean_tweet'].iloc[i][:80]}")
    print()

Dataset after cleaning: 24780 rows

Before - After examples:
  Original:  !!! RT @mayasolovely: As a woman you shouldn't complain about cleaning up your h...
  Cleaned:   woman shouldnt complain cleaning house man always take trash

  Original:  "@ClicquotSuave: LMAOOOOOOOOOOO this nigga @Krillz_Nuh_Care http://t.co/AAnpSUjm...
  Cleaned:   lmaooooooooooo nigga bitch want like depressing shitfoh

  Original:  "I'm probably your main bitch chocolate dipped cinnamon apple"...
  Cleaned:   probably main bitch chocolate dipped cinnamon apple



In [10]:
X = df['clean_tweet']
y = df['class']

X_train, X_temp, y_train, y_temp = train_test_split(
    X, y, test_size=0.30, random_state=42, stratify=y
)

X_val, X_test, y_val, y_test = train_test_split(
    X_temp, y_temp, test_size=0.50, random_state=42, stratify=y_temp
)

tfidf = TfidfVectorizer(
    max_features=30000,
    ngram_range=(1, 2),
    min_df=2,
    max_df=0.95,
    sublinear_tf=True
)

X_train_tfidf = tfidf.fit_transform(X_train)
X_val_tfidf = tfidf.transform(X_val)
X_test_tfidf = tfidf.transform(X_test)

print(f"Train: {X_train_tfidf.shape}")
print(f"Validation: {X_val_tfidf.shape}")
print(f"Test: {X_test_tfidf.shape}")
print(f"Vocabulary size: {len(tfidf.vocabulary_):,}")

Train: (17346, 16214)
Validation: (3717, 16214)
Test: (3717, 16214)
Vocabulary size: 16,214


## Algorithmes retenus et justification
Nous comparons 4 algorithmes de classification, chacun avec un réglage d'hyperparamètres via `RandomizedSearchCV` :

1. **Régression logistique** : baseline robuste pour le texte TF-IDF; interprétable et efficace sur grande dimension.
2. **SVM linéaire** : très performant sur les représentations creuses du NLP.
3. **Naive Bayes multinomial** : modèle rapide, adapté aux fréquences de mots et aux features TF-IDF.
4. **Random Forest** : modèle non-linéaire pour capter des interactions entre features.

Chaque modèle est ajusté sur l'entraînement puis comparé sur validation avant test final.

In [11]:
search_spaces = {
    "Logistic Regression": {
        'estimator': LogisticRegression(solver='saga', max_iter=1500, random_state=42, n_jobs=-1),
        'params': {
            'C': loguniform(1e-2, 20),
            'penalty': ['l1', 'l2']
        }
    },
    "Linear SVM": {
        'estimator': LinearSVC(random_state=42),
        'params': {
            'C': loguniform(1e-3, 30),
            'max_iter': [2000, 3000, 5000]
        }
    },
    "Multinomial NB": {
        'estimator': MultinomialNB(),
        'params': {
            'alpha': loguniform(1e-3, 2)
        }
    },
    "Random Forest": {
        'estimator': RandomForestClassifier(random_state=42, n_jobs=-1),
        'params': {
            'n_estimators': randint(150, 500),
            'max_depth': [None, 20, 40, 60],
            'min_samples_split': randint(2, 10),
            'min_samples_leaf': randint(1, 5)
        }
    }
}

best_models = {}
results = []

print(f"{'Model':<22} {'Val Acc':>10} {'Val Macro-F1':>14} {'Best Params':>20}")
print("=" * 90)

for name, spec in search_spaces.items():
    search = RandomizedSearchCV(
        estimator=spec["estimator"],
        param_distributions=spec["params"],
        n_iter=10,
        scoring="f1_macro",
        cv=3,
        random_state=42,
        n_jobs=-1,
        verbose=0
    )
    search.fit(X_train_tfidf, y_train)

    best_model = search.best_estimator_

    # Calibrage pour obtenir predict_proba quand le modèle ne l'expose pas nativement
    if not hasattr(best_model, "predict_proba"):
        calibrated = CalibratedClassifierCV(
            estimator=clone(best_model),
            method="sigmoid",
            cv=3
        )
        calibrated.fit(X_train_tfidf, y_train)
        best_model = calibrated

    val_pred = best_model.predict(X_val_tfidf)
    val_acc = accuracy_score(y_val, val_pred)
    val_f1 = f1_score(y_val, val_pred, average="macro")

    best_models[name] = best_model
    results.append({
        "Model": name,
        "Val Accuracy": val_acc,
        "Val Macro F1": val_f1,
        "Best Params": search.best_params_
    })

    print(f"{name:<22} {val_acc:>10.4f} {val_f1:>14.4f} {str(search.best_params_)[:20]:>20}")

print("\nHyperparameter tuning complete.")

Model                     Val Acc   Val Macro-F1          Best Params
Logistic Regression        0.9023         0.7103 {'C': np.float64(5.5
Linear SVM                 0.8991         0.6901 {'C': np.float64(1.7
Multinomial NB             0.8579         0.6105 {'alpha': np.float64
Random Forest              0.8956         0.6340 {'max_depth': None, 

Hyperparameter tuning complete.


## DistilBERT (configuration optimale projet)

Ajout d'un entraînement DistilBERT avec la meilleure configuration observée dans `notebook_principal.ipynb` et les rapports du projet (`run_d_balanced`).
Cette cellule ajoute DistilBERT à `best_models` et `results` pour l'intégrer automatiquement aux comparaisons, rapports de classification et matrices de confusion.

In [12]:
try:
    from model_zoo import OptimizedDistilBertClassifier, distilbert_deps_available
except ModuleNotFoundError:
    from Code.model_zoo import OptimizedDistilBertClassifier, distilbert_deps_available

In [13]:
DISTILBERT_BEST_CONFIG = {
    "model_name": "distilbert-base-uncased",
    "epochs": 2,
    "batch_size": 8,
    "max_length": 160,
    "learning_rate": 3e-5,
    "weight_decay": 0.01,
    "random_state": 42,
    "warmup_ratio": 0.1,
    "focal_gamma": 1.5,
    "use_balanced_sampler": True,
    "enable_error_driven_finetune": True,
    "error_driven_repeat": 2,
}

if distilbert_deps_available():
    print("Training DistilBERT with optimized pipeline (weights/focal/sampler/warmup/early-stop/threshold/error-driven)...")
    distilbert_model = OptimizedDistilBertClassifier(**DISTILBERT_BEST_CONFIG)
    distilbert_model.fit(X_train, y_train, x_val=X_val, y_val=y_val)

    val_pred_distilbert = distilbert_model.predict(X_val)
    val_acc_distilbert = accuracy_score(y_val, val_pred_distilbert)
    val_f1_distilbert = f1_score(y_val, val_pred_distilbert, average="macro")

    best_models["DistilBERT"] = distilbert_model
    results.append(
        {
            "Model": "DistilBERT",
            "Val Accuracy": val_acc_distilbert,
            "Val Macro F1": val_f1_distilbert,
            "Best Params": {
                "epochs": DISTILBERT_BEST_CONFIG["epochs"],
                "batch_size": DISTILBERT_BEST_CONFIG["batch_size"],
                "max_length": DISTILBERT_BEST_CONFIG["max_length"],
                "learning_rate": DISTILBERT_BEST_CONFIG["learning_rate"],
                "weight_decay": DISTILBERT_BEST_CONFIG["weight_decay"],
                "warmup_ratio": DISTILBERT_BEST_CONFIG["warmup_ratio"],
                "focal_gamma": DISTILBERT_BEST_CONFIG["focal_gamma"],
                "balanced_sampler": DISTILBERT_BEST_CONFIG["use_balanced_sampler"],
                "hate_threshold": round(distilbert_model.hate_threshold, 3),
            },
        }
    )

    print(
        f"DistilBERT added | Val Acc: {val_acc_distilbert:.4f} | "
        f"Val Macro-F1: {val_f1_distilbert:.4f} | "
        f"Hate threshold: {distilbert_model.hate_threshold:.2f}"
    )
else:
    print("DistilBERT skipped: missing torch/transformers/datasets dependencies.")

Some weights of DistilBertForSequenceClassification were not initialized from the model checkpoint at distilbert-base-uncased and are newly initialized: ['classifier.bias', 'classifier.weight', 'pre_classifier.bias', 'pre_classifier.weight']
You should probably TRAIN this model on a down-stream task to be able to use it for predictions and inference.


Training DistilBERT with optimized pipeline (weights/focal/sampler/warmup/early-stop/threshold/error-driven)...


Map: 100%|██████████| 3717/3717 [00:00<00:00, 153589.23 examples/s]


Epoch,Training Loss,Validation Loss,F1 Macro,Recall Macro,Hate Recall
1,0.137300,0.380756,0.686335,0.816581,0.761682
2,0.039100,0.560729,0.738759,0.809857,0.626168


Map: 100%|██████████| 17506/17506 [00:00<00:00, 198296.66 examples/s]


Epoch,Training Loss,Validation Loss,F1 Macro,Recall Macro,Hate Recall
1,0.108100,0.213942,0.814667,0.912797,0.925234
2,0.017800,0.268139,0.824434,0.897224,0.869159


Map: 100%|██████████| 3717/3717 [00:00<00:00, 179944.69 examples/s]


DistilBERT added | Val Acc: 0.9104 | Val Macro-F1: 0.8244 | Hate threshold: 0.50


In [14]:
res_df = pd.DataFrame(results).sort_values('Val Macro F1', ascending=False).reset_index(drop=True)

print("\n" + "=" * 95)
print("                     MODEL RANKING ON VALIDATION SET")
print("=" * 95)
print(res_df[['Model', 'Val Accuracy', 'Val Macro F1']].to_string(index=False))
print("=" * 95)

best_model_name = res_df.iloc[0]['Model']
best_model = best_models[best_model_name]

print(f"\nBest model selected from validation: {best_model_name}")
print(f"Best hyperparameters: {res_df.iloc[0]['Best Params']}")


                     MODEL RANKING ON VALIDATION SET
              Model  Val Accuracy  Val Macro F1
         DistilBERT      0.910412      0.824434
Logistic Regression      0.902341      0.710308
         Linear SVM      0.899112      0.690066
      Random Forest      0.895615      0.634035
     Multinomial NB      0.857950      0.610500

Best model selected from validation: DistilBERT
Best hyperparameters: {'epochs': 2, 'batch_size': 8, 'max_length': 160, 'learning_rate': 3e-05, 'weight_decay': 0.01, 'warmup_ratio': 0.1, 'focal_gamma': 1.5, 'balanced_sampler': True, 'hate_threshold': 0.5}


In [15]:
# Unified evaluation: DistilBERT uses raw text, classic models use TF-IDF features
test_results = []
pred_store = {}
for name, model in best_models.items():
    if name == "DistilBERT" or getattr(model, "is_deep_model", False):
        # DistilBERT expects raw text, not sparse TF-IDF matrix
        y_pred = model.predict(X_test)
    else:
        # Classic sklearn models expect TF-IDF-transformed features
        y_pred = model.predict(X_test_tfidf)
    pred_store[name] = y_pred
    precision, recall, f1, _ = precision_recall_fscore_support(
        y_test, y_pred, average="macro", zero_division=0
    )
    test_results.append(
        {
            "Model": name,
            "Test Accuracy": accuracy_score(y_test, y_pred),
            "Test Precision (macro)": precision,
            "Test Recall (macro)": recall,
            "Test F1 (macro)": f1,
        }
    )
test_df_metrics = (
    pd.DataFrame(test_results)
    .sort_values("Test F1 (macro)", ascending=False)
    .reset_index(drop=True)
)
print(test_df_metrics.to_string(index=False))

Map: 100%|██████████| 3717/3717 [00:00<00:00, 185768.24 examples/s]


              Model  Test Accuracy  Test Precision (macro)  Test Recall (macro)  Test F1 (macro)
         DistilBERT       0.890234                0.728727             0.790821         0.754824
Logistic Regression       0.895615                0.775013             0.682977         0.711066
         Linear SVM       0.890503                0.777064             0.644375         0.672425
      Random Forest       0.885660                0.798384             0.605112         0.618600
     Multinomial NB       0.859026                0.747345             0.576794         0.616171


In [16]:
fig, axes = plt.subplots(1, 2, figsize=(16, 6))

sorted_res = test_df_metrics.sort_values('Test Accuracy', ascending=True)
colors = plt.cm.RdYlGn(np.linspace(0.2, 0.9, len(sorted_res)))

axes[0].barh(sorted_res['Model'], sorted_res['Test Accuracy'], color=colors, edgecolor='black')
axes[0].set_xlabel('Accuracy', fontsize=12)
axes[0].set_title('Test Accuracy Comparison', fontsize=14, fontweight='bold')
axes[0].set_xlim(0.0, 1.0)
for i, acc in enumerate(sorted_res['Test Accuracy']):
    axes[0].text(min(acc + 0.01, 0.98), i, f'{acc:.4f}', va='center', fontsize=10, fontweight='bold')

axes[1].barh(sorted_res['Model'], sorted_res['Test F1 (macro)'], color=colors, edgecolor='black')
axes[1].set_xlabel('Macro F1', fontsize=12)
axes[1].set_title('Test Macro-F1 Comparison', fontsize=14, fontweight='bold')
axes[1].set_xlim(0.0, 1.0)
for i, f1 in enumerate(sorted_res['Test F1 (macro)']):
    axes[1].text(min(f1 + 0.01, 0.98), i, f'{f1:.4f}', va='center', fontsize=10, fontweight='bold')

plt.tight_layout()
plt.show()

In [17]:
target_names = ['Hate Speech (0)', 'Offensive (1)', 'Neither (2)']

for name in test_df_metrics['Model']:
    print(f"\nClassification Report - {name}")
    print("=" * 70)
    print(classification_report(y_test, pred_store[name], target_names=target_names, zero_division=0))


Classification Report - DistilBERT
                 precision    recall  f1-score   support

Hate Speech (0)       0.39      0.54      0.46       215
  Offensive (1)       0.96      0.91      0.93      2878
    Neither (2)       0.84      0.92      0.87       624

       accuracy                           0.89      3717
      macro avg       0.73      0.79      0.75      3717
   weighted avg       0.90      0.89      0.90      3717


Classification Report - Logistic Regression
                 precision    recall  f1-score   support

Hate Speech (0)       0.57      0.27      0.37       215
  Offensive (1)       0.92      0.96      0.94      2878
    Neither (2)       0.84      0.82      0.83       624

       accuracy                           0.90      3717
      macro avg       0.78      0.68      0.71      3717
   weighted avg       0.88      0.90      0.89      3717


Classification Report - Linear SVM
                 precision    recall  f1-score   support

Hate Speech (0)      

In [18]:
models_for_cm = list(test_df_metrics['Model'])
n_models = len(models_for_cm)
fig, axes = plt.subplots(2, n_models, figsize=(4 * n_models, 8))

if n_models == 1:
    axes = np.array(axes).reshape(2, 1)

for i, name in enumerate(models_for_cm):
    y_pred = pred_store[name]

    cm = confusion_matrix(y_test, y_pred)
    disp_counts = ConfusionMatrixDisplay(cm, display_labels=target_names)
    disp_counts.plot(ax=axes[0, i], cmap='Reds', values_format='d', colorbar=False)
    axes[0, i].set_title(f'{name}\nCounts', fontsize=11, fontweight='bold')
    axes[0, i].tick_params(axis='x', rotation=20)

    cm_norm = confusion_matrix(y_test, y_pred, normalize='true')
    disp_norm = ConfusionMatrixDisplay(cm_norm, display_labels=target_names)
    disp_norm.plot(ax=axes[1, i], cmap='Blues', values_format='.2f', colorbar=False)
    axes[1, i].set_title(f'{name}\nNormalized', fontsize=11, fontweight='bold')
    axes[1, i].tick_params(axis='x', rotation=20)

plt.tight_layout()
plt.show()

In [19]:
# 5-fold cross-validation for each tuned model (DistilBERT = true stratified CV on raw text)
X_trainval = pd.concat([X_train, X_val]).reset_index(drop=True)
y_trainval = pd.concat([y_train, y_val]).reset_index(drop=True).astype(int)
X_trainval_tfidf = tfidf.transform(X_trainval)

skf = StratifiedKFold(n_splits=5, shuffle=True, random_state=42)
cv_rows = []

print("5-Fold Stratified Cross-Validation (Accuracy & Macro-F1)")
print("=" * 80)

for name, model in best_models.items():
    if name == "DistilBERT" or getattr(model, "is_deep_model", False):
        # True CV for DistilBERT on raw text
        fold_acc, fold_f1 = [], []

        for fold, (tr_idx, va_idx) in enumerate(skf.split(X_trainval, y_trainval), start=1):
            X_tr = X_trainval.iloc[tr_idx]
            y_tr = y_trainval.iloc[tr_idx]
            X_va = X_trainval.iloc[va_idx]
            y_va = y_trainval.iloc[va_idx]

            # Re-instancie un modèle propre à chaque fold
            model_cv = OptimizedDistilBertClassifier(
                model_name=getattr(model, "model_name", "distilbert-base-uncased"),
                max_length=getattr(model, "max_length", 160),
                epochs=getattr(model, "epochs", 2),
                batch_size=getattr(model, "batch_size", 16),
                learning_rate=getattr(model, "learning_rate", 3e-5),
                weight_decay=getattr(model, "weight_decay", 0.01),
                random_state=42 + fold,  # seed fold
                warmup_ratio=getattr(model, "warmup_ratio", 0.1),
                focal_gamma=getattr(model, "focal_gamma", 1.5),
                use_balanced_sampler=getattr(model, "use_balanced_sampler", True),
                enable_error_driven_finetune=getattr(model, "enable_error_driven_finetune", True),
                error_driven_repeat=getattr(model, "error_driven_repeat", 2),
            )

            model_cv.fit(X_tr, y_tr, x_val=X_va, y_val=y_va)
            y_pred = model_cv.predict(X_va)

            fold_acc.append(accuracy_score(y_va, y_pred))
            fold_f1.append(f1_score(y_va, y_pred, average="macro"))

        cv_rows.append({
            "Model": name,
            "CV Accuracy Mean": float(np.mean(fold_acc)),
            "CV Accuracy Std": float(np.std(fold_acc)),
            "CV F1 Mean": float(np.mean(fold_f1)),
            "CV F1 Std": float(np.std(fold_f1)),
            "CV Source": "kfold_raw_text_distilbert",
        })

        print(
            f"{name:<22} | "
            f"Acc: {np.mean(fold_acc):.4f} +/- {np.std(fold_acc):.4f} | "
            f"F1: {np.mean(fold_f1):.4f} +/- {np.std(fold_f1):.4f}"
        )
        continue

    # Classic models: CV on TF-IDF features
    acc_scores = cross_val_score(
        model, X_trainval_tfidf, y_trainval, cv=skf, scoring="accuracy", n_jobs=-1
    )
    f1_scores = cross_val_score(
        model, X_trainval_tfidf, y_trainval, cv=skf, scoring="f1_macro", n_jobs=-1
    )

    cv_rows.append({
        "Model": name,
        "CV Accuracy Mean": float(acc_scores.mean()),
        "CV Accuracy Std": float(acc_scores.std()),
        "CV F1 Mean": float(f1_scores.mean()),
        "CV F1 Std": float(f1_scores.std()),
        "CV Source": "kfold_tfidf",
    })

    print(
        f"{name:<22} | "
        f"Acc: {acc_scores.mean():.4f} +/- {acc_scores.std():.4f} | "
        f"F1: {f1_scores.mean():.4f} +/- {f1_scores.std():.4f}"
    )

cv_df = (
    pd.DataFrame(cv_rows)
    .sort_values("CV F1 Mean", ascending=False)
    .reset_index(drop=True)
)

display(cv_df)

5-Fold Stratified Cross-Validation (Accuracy & Macro-F1)
Logistic Regression    | Acc: 0.8990 +/- 0.0022 | F1: 0.7035 +/- 0.0116
Linear SVM             | Acc: 0.8957 +/- 0.0025 | F1: 0.6763 +/- 0.0123
Multinomial NB         | Acc: 0.8543 +/- 0.0050 | F1: 0.6180 +/- 0.0105


Some weights of DistilBertForSequenceClassification were not initialized from the model checkpoint at distilbert-base-uncased and are newly initialized: ['classifier.bias', 'classifier.weight', 'pre_classifier.bias', 'pre_classifier.weight']
You should probably TRAIN this model on a down-stream task to be able to use it for predictions and inference.


Random Forest          | Acc: 0.8937 +/- 0.0027 | F1: 0.6284 +/- 0.0112


Map: 100%|██████████| 4213/4213 [00:00<00:00, 190652.24 examples/s]


Epoch,Training Loss,Validation Loss,F1 Macro,Recall Macro,Hate Recall
1,0.457400,0.353467,0.695841,0.830233,0.802469
2,0.062400,0.543855,0.749238,0.816040,0.625514


...

Map: 100%|██████████| 17071/17071 [00:00<00:00, 185934.51 examples/s]


Epoch,Training Loss,Validation Loss,F1 Macro,Recall Macro,Hate Recall
1,0.089500,0.231711,0.812347,0.914934,0.913580
2,0.132300,0.304579,0.822861,0.883191,0.818930


Map: 100%|██████████| 4212/4212 [00:00<00:00, 174417.59 examples/s]


DistilBERT             | Acc: 0.9120 +/- 0.0063 | F1: 0.8279 +/- 0.0121


,Model,CV Accuracy Mean,CV Accuracy Std,CV F1 Mean,CV F1 Std,CV Source
0,DistilBERT,0.911979,0.006324,0.827945,0.012118,kfold_raw_text_distilbert
1,Logistic Regression,0.899017,0.002220,0.703538,0.011625,kfold_tfidf
2,Linear SVM,0.895741,0.002484,0.676297,0.012270,kfold_tfidf
3,Random Forest,0.893700,0.002724,0.628431,0.011217,kfold_tfidf
4,Multinomial NB,0.854294,0.004977,0.617952,0.010524,kfold_tfidf


In [20]:
# Learning curve for the selected final model
learning_model_name = best_model_name
learning_model = best_model

if best_model_name == 'DistilBERT' or getattr(best_model, 'is_deep_model', False):
    print('Learning curve strategy: progressive DistilBERT re-training on raw text.')

    fractions = np.linspace(0.2, 1.0, 5)
    lc_train_sizes = []
    lc_train_f1 = []
    lc_val_f1 = []

    for frac in fractions:
        if frac < 1.0:
            X_sub, _, y_sub, _ = train_test_split(
                X_train,
                y_train,
                train_size=float(frac),
                random_state=42,
                stratify=y_train,
            )
        else:
            X_sub, y_sub = X_train, y_train

        model_lc = clone(best_model)
        model_lc.fit(X_sub, y_sub, x_val=X_val, y_val=y_val)

        y_sub_pred = model_lc.predict(X_sub)
        y_val_pred = model_lc.predict(X_val)

        lc_train_sizes.append(len(X_sub))
        lc_train_f1.append(f1_score(y_sub, y_sub_pred, average='macro'))
        lc_val_f1.append(f1_score(y_val, y_val_pred, average='macro'))

        print(
            f"  size={len(X_sub):5d} | "
            f"train_f1={lc_train_f1[-1]:.4f} | "
            f"val_f1={lc_val_f1[-1]:.4f}"
        )

    plt.figure(figsize=(8, 5))
    plt.plot(lc_train_sizes, lc_train_f1, marker='o', label='Train F1 (macro)')
    plt.plot(lc_train_sizes, lc_val_f1, marker='s', label='Validation F1 (macro)')
    plt.title('Learning Curve - DistilBERT', fontweight='bold')
    plt.xlabel('Training set size')
    plt.ylabel('Macro F1')
    plt.legend()
    plt.grid(alpha=0.2)
    plt.tight_layout()
    plt.show()

else:
    train_sizes, train_scores, val_scores = learning_curve(
        estimator=learning_model,
        X=X_trainval_tfidf,
        y=y_trainval,
        train_sizes=np.linspace(0.2, 1.0, 5),
        cv=3,
        scoring='f1_macro',
        n_jobs=-1
    )

    plt.figure(figsize=(8, 5))
    plt.plot(train_sizes, train_scores.mean(axis=1), marker='o', label='Train F1 (macro)')
    plt.plot(train_sizes, val_scores.mean(axis=1), marker='s', label='Validation F1 (macro)')
    plt.fill_between(
        train_sizes,
        val_scores.mean(axis=1) - val_scores.std(axis=1),
        val_scores.mean(axis=1) + val_scores.std(axis=1),
        alpha=0.2
    )
    plt.title(f'Learning Curve - {learning_model_name}', fontweight='bold')
    plt.xlabel('Training set size')
    plt.ylabel('Macro F1')
    plt.legend()
    plt.grid(alpha=0.2)
    plt.tight_layout()
    plt.show()

Some weights of DistilBertForSequenceClassification were not initialized from the model checkpoint at distilbert-base-uncased and are newly initialized: ['classifier.bias', 'classifier.weight', 'pre_classifier.bias', 'pre_classifier.weight']
You should probably TRAIN this model on a down-stream task to be able to use it for predictions and inference.


Learning curve strategy: progressive DistilBERT re-training on raw text.


Map: 100%|██████████| 3717/3717 [00:00<00:00, 161766.31 examples/s]


Epoch,Training Loss,Validation Loss,F1 Macro,Recall Macro,Hate Recall
1,0.247900,0.427927,0.711961,0.788473,0.584112
2,0.073200,0.584508,0.719775,0.782058,0.556075


Map: 100%|██████████| 3659/3659 [00:00<00:00, 177224.79 examples/s]


...

Map: 100%|██████████| 17506/17506 [00:00<00:00, 186304.72 examples/s]


Epoch,Training Loss,Validation Loss,F1 Macro,Recall Macro,Hate Recall
1,0.108100,0.213942,0.814667,0.912797,0.925234
2,0.017800,0.268139,0.824434,0.897224,0.869159


Map: 100%|██████████| 17346/17346 [00:00<00:00, 186837.18 examples/s]


Map: 100%|██████████| 3717/3717 [00:00<00:00, 165640.27 examples/s]


  size=17346 | train_f1=0.9030 | val_f1=0.8244


In [21]:
# Feature importance / influential terms
feature_names = np.array(tfidf.get_feature_names_out())

feature_model_name = best_model_name
feature_model = best_model
if best_model_name == 'DistilBERT':
    fallback = test_df_metrics[test_df_metrics['Model'] != 'DistilBERT']
    if not fallback.empty:
        feature_model_name = fallback.sort_values('Test F1 (macro)', ascending=False).iloc[0]['Model']
        feature_model = best_models[feature_model_name]
        print(f"Feature importance fallback (TF-IDF model): {feature_model_name}")

if hasattr(feature_model, 'coef_'):
    coef = feature_model.coef_
    importance = np.mean(np.abs(coef), axis=0)
    top_idx = np.argsort(importance)[-20:]
    top_values = importance[top_idx]
    top_labels = feature_names[top_idx]
    title = f'Top 20 Influential Features - {feature_model_name} (|coef| mean)'
elif hasattr(feature_model, 'feature_importances_'):
    importance = feature_model.feature_importances_
    top_idx = np.argsort(importance)[-20:]
    top_values = importance[top_idx]
    top_labels = feature_names[top_idx]
    title = f'Top 20 Feature Importances - {feature_model_name}'
else:
    top_idx = np.array([], dtype=int)
    top_values = np.array([])
    top_labels = np.array([])
    title = f'Feature importance not available for {feature_model_name}'

if len(top_idx) > 0:
    plt.figure(figsize=(9, 7))
    plt.barh(top_labels, top_values, color='#3498db', edgecolor='black')
    plt.title(title, fontweight='bold')
    plt.xlabel('Importance score')
    plt.tight_layout()
    plt.show()
else:
    print(title)

Feature importance fallback (TF-IDF model): Logistic Regression


In [22]:
best_preds = pred_store[best_model_name]

analysis_df = pd.DataFrame({
    'tweet': X_test.values,
    'true_label': y_test.values,
    'predicted': best_preds
})
analysis_df['correct'] = analysis_df['true_label'] == analysis_df['predicted']

misclassified = analysis_df[~analysis_df['correct']].copy()
misclassified['true_name'] = misclassified['true_label'].map(label_map)
misclassified['pred_name'] = misclassified['predicted'].map(label_map)

print(f"Final model analyzed: {best_model_name}")
print(f"Total test samples: {len(analysis_df):,}")
print(f"Misclassified: {len(misclassified):,} ({len(misclassified)/len(analysis_df)*100:.1f}%)")
print()

error_types = misclassified.groupby(['true_name', 'pred_name']).size().sort_values(ascending=False)
print("Most common error types:")
print(error_types.head(6).to_string())

print("\n--- Sample Misclassified Tweets ---")
for _, row in misclassified.head(5).iterrows():
    print(f"  True: {row['true_name']:<20} Pred: {row['pred_name']:<20}")
    print(f"  Tweet: {row['tweet'][:100]}...")
    print()

Final model analyzed: DistilBERT
Total test samples: 3,717
Misclassified: 408 (11.0%)

Most common error types:
true_name           pred_name         
Offensive Language  Hate Speech           165
                    Neither                94
Hate Speech         Offensive Language     79
Neither             Offensive Language     35
Hate Speech         Neither                19
Neither             Hate Speech            16

--- Sample Misclassified Tweets ---
  True: Offensive Language   Pred: Hate Speech         
  Tweet: lmao hate hoe already...

  True: Hate Speech          Pred: Neither             
  Tweet: guess blaspheme allah po muzzie huh...

  True: Hate Speech          Pred: Offensive Language  
  Tweet: lrg as nigguh...

  True: Hate Speech          Pred: Offensive Language  
  Tweet: want chill campus throw bible bitch taking walk shame...

  True: Offensive Language   Pred: Neither             
  Tweet: agree northkorea say barackobama look like monkey belongs african zoo

In [23]:
import json, joblib

# Refit the selected model on train+validation before final saving
X_trainval = pd.concat([X_train, X_val])
y_trainval = pd.concat([y_train, y_val])

best_for_save = best_models[best_model_name]

if best_model_name == "DistilBERT" or getattr(best_for_save, "is_deep_model", False):
    best_for_save.fit(X_trainval, y_trainval)
    export_dir = Path("distilbert_export")
    export_dir.mkdir(parents=True, exist_ok=True)
    # modèle/tokenizer HF
    best_for_save._trainer.model.save_pretrained(export_dir)
    best_for_save._tokenizer.save_pretrained(export_dir)
    # métadonnées utiles à l'inférence
    (export_dir / "meta.json").write_text(json.dumps({
        "best_model_name": best_model_name,
        "label_to_id": best_for_save.label_to_id,
        "id_to_label": best_for_save.id_to_label,
        "max_length": best_for_save.max_length,
        "hate_threshold": getattr(best_for_save, "hate_threshold", 0.5),
        "threshold_metrics": getattr(best_for_save, "threshold_metrics", {}),
    }, indent=2), encoding="utf-8")
    print(f"DistilBERT saved to: {export_dir.resolve()}")
else:
    # seulement modèles sklearn/classiques
    best_for_save.fit(X_trainval_tfidf, y_trainval)
    joblib.dump(
        {"model": best_for_save, "tfidf": tfidf, "label_map": label_map},
        "hate_speech_model.pkl"
    )
    print("Saved: hate_speech_model.pkl")

Some weights of DistilBertForSequenceClassification were not initialized from the model checkpoint at distilbert-base-uncased and are newly initialized: ['classifier.bias', 'classifier.weight', 'pre_classifier.bias', 'pre_classifier.weight']
You should probably TRAIN this model on a down-stream task to be able to use it for predictions and inference.
Map: 100%|██████████| 21063/21063 [00:00<00:00, 185597.17 examples/s]


Step,Training Loss
50,1.654800
100,1.201300
150,0.921300
200,0.732600
250,0.538600
300,0.632800
350,0.411600
400,0.584200
450,0.477500
500,0.510200


DistilBERT saved to: /workspace/distilbert_export


In [24]:

def predict_hate_speech(texts, model=best_for_save, vectorizer=tfidf):
    if isinstance(texts, str):
        texts = [texts]

    cleaned = [preprocess_text(t) for t in texts]
    is_deep = getattr(model, "is_deep_model", False)

    # Predict labels
    if is_deep:
        preds = model.predict(cleaned)
        probas = model.predict_proba(cleaned) if hasattr(model, "predict_proba") else None
    else:
        features = vectorizer.transform(cleaned)
        preds = model.predict(features)
        probas = model.predict_proba(features) if hasattr(model, "predict_proba") else None

    rows = []
    for i, (text, pred) in enumerate(zip(texts, preds)):
        row = {
            "text": text[:80],
            "prediction": label_map[int(pred)],
            "confidence": None,
            "probabilities": None,
        }

        if probas is not None:
            probs_i = np.asarray(probas[i], dtype=float)
            row["confidence"] = f"{probs_i.max() * 100:.1f}%"
            row["probabilities"] = {label_map[j]: f"{p:.3f}" for j, p in enumerate(probs_i)}
        elif is_deep:
            row["confidence"] = "thresholded"

        rows.append(row)

    return rows


test_texts = [
    "I love spending time with my friends at the park",
    "This movie is absolutely terrible, worst thing ever made",
    "You're all stupid idiots who deserve nothing",
    "Great game today! What a performance by the team",
    "I hate mondays so much, worst day of the week",
]

print("Quick Inference Demo")
print("=" * 70)
for r in predict_hate_speech(test_texts):
    print(f"Text: {r['text']}")
    print(f"-> {r['prediction']}")
    if r.get("confidence") is not None:
        print(f"   Confidence: {r['confidence']}")
    if r.get("probabilities") is not None:
        print(f"   Probabilities: {r['probabilities']}")
    print()

Quick Inference Demo


Map: 100%|██████████| 5/5 [00:00<00:00, 2809.31 examples/s]


Map: 100%|██████████| 5/5 [00:00<00:00, 2777.68 examples/s]


Text: I love spending time with my friends at the park
-> Neither
   Confidence: 94.3%
   Probabilities: {'Hate Speech': '0.003', 'Offensive Language': '0.054', 'Neither': '0.943'}

Text: This movie is absolutely terrible, worst thing ever made
-> Neither
   Confidence: 96.3%
   Probabilities: {'Hate Speech': '0.003', 'Offensive Language': '0.033', 'Neither': '0.963'}

Text: You're all stupid idiots who deserve nothing
-> Hate Speech
   Confidence: 96.1%
   Probabilities: {'Hate Speech': '0.961', 'Offensive Language': '0.034', 'Neither': '0.005'}

Text: Great game today! What a performance by the team
-> Neither
   Confidence: 97.3%
   Probabilities: {'Hate Speech': '0.003', 'Offensive Language': '0.024', 'Neither': '0.973'}

Text: I hate mondays so much, worst day of the week
-> Neither
   Confidence: 94.8%
   Probabilities: {'Hate Speech': '0.004', 'Offensive Language': '0.048', 'Neither': '0.948'}



In [25]:
import os
from datetime import datetime

project_root = Path.cwd()          
output_dir = project_root / "Outputs"
output_dir.mkdir(parents=True, exist_ok=True)

# 1. Sauvegarde des DataFrames résultats
res_df[['Model', 'Val Accuracy', 'Val Macro F1']].to_csv(f'{output_dir}/validation_results.csv', index=False)
test_df_metrics.to_csv(f'{output_dir}/test_results.csv', index=False)
cv_df.to_csv(f'{output_dir}/cross_validation_results.csv', index=False)

# 2. Sauvegarde des diagrammes
# Diagramme: Comparaison performance
fig_perf, axes = plt.subplots(1, 2, figsize=(16, 6))
sorted_res = test_df_metrics.sort_values('Test Accuracy', ascending=True)
colors = plt.cm.RdYlGn(np.linspace(0.2, 0.9, len(sorted_res)))

axes[0].barh(sorted_res['Model'], sorted_res['Test Accuracy'], color=colors, edgecolor='black')
axes[0].set_xlabel('Accuracy', fontsize=12)
axes[0].set_title('Test Accuracy Comparison', fontsize=14, fontweight='bold')
axes[0].set_xlim(0.0, 1.0)
for i, acc in enumerate(sorted_res['Test Accuracy']):
    axes[0].text(min(acc + 0.01, 0.98), i, f'{acc:.4f}', va='center', fontsize=10, fontweight='bold')

axes[1].barh(sorted_res['Model'], sorted_res['Test F1 (macro)'], color=colors, edgecolor='black')
axes[1].set_xlabel('Macro F1', fontsize=12)
axes[1].set_title('Test Macro-F1 Comparison', fontsize=14, fontweight='bold')
axes[1].set_xlim(0.0, 1.0)
for i, f1 in enumerate(sorted_res['Test F1 (macro)']):
    axes[1].text(min(f1 + 0.01, 0.98), i, f'{f1:.4f}', va='center', fontsize=10, fontweight='bold')

plt.tight_layout()
fig_perf.savefig(f'{output_dir}/01_model_comparison.png', dpi=300, bbox_inches='tight')
plt.close()

# Diagrammes: Matrices de confusion
models_for_cm = list(test_df_metrics['Model'])
n_models = len(models_for_cm)
target_names = ['Hate Speech (0)', 'Offensive (1)', 'Neither (2)']

fig_cm, axes = plt.subplots(2, n_models, figsize=(4 * n_models, 8))
if n_models == 1:
    axes = np.array(axes).reshape(2, 1)

for i, name in enumerate(models_for_cm):
    y_pred = pred_store[name]

    cm = confusion_matrix(y_test, y_pred)
    disp_counts = ConfusionMatrixDisplay(cm, display_labels=target_names)
    disp_counts.plot(ax=axes[0, i], cmap='Reds', values_format='d', colorbar=False)
    axes[0, i].set_title(f'{name}\nCounts', fontsize=11, fontweight='bold')
    axes[0, i].tick_params(axis='x', rotation=20)

    cm_norm = confusion_matrix(y_test, y_pred, normalize='true')
    disp_norm = ConfusionMatrixDisplay(cm_norm, display_labels=target_names)
    disp_norm.plot(ax=axes[1, i], cmap='Blues', values_format='.2f', colorbar=False)
    axes[1, i].set_title(f'{name}\nNormalized', fontsize=11, fontweight='bold')
    axes[1, i].tick_params(axis='x', rotation=20)

plt.tight_layout()
fig_cm.savefig(f'{output_dir}/02_confusion_matrices.png', dpi=300, bbox_inches='tight')
plt.close()

# Diagramme: Courbe d'apprentissage
learning_model_name = best_model_name
learning_model = best_model

if best_model_name == 'DistilBERT' or getattr(best_model, 'is_deep_model', False):
    print('Learning curve strategy: progressive DistilBERT re-training on raw text.')

    fractions = np.linspace(0.2, 1.0, 5)
    lc_train_sizes = []
    lc_train_f1 = []
    lc_val_f1 = []

    for frac in fractions:
        if frac < 1.0:
            X_sub, _, y_sub, _ = train_test_split(
                X_train,
                y_train,
                train_size=float(frac),
                random_state=42,
                stratify=y_train,
            )
        else:
            X_sub, y_sub = X_train, y_train

        model_lc = clone(best_model)
        model_lc.fit(X_sub, y_sub, x_val=X_val, y_val=y_val)

        y_sub_pred = model_lc.predict(X_sub)
        y_val_pred = model_lc.predict(X_val)

        lc_train_sizes.append(len(X_sub))
        lc_train_f1.append(f1_score(y_sub, y_sub_pred, average='macro'))
        lc_val_f1.append(f1_score(y_val, y_val_pred, average='macro'))

        print(
            f"  size={len(X_sub):5d} | "
            f"train_f1={lc_train_f1[-1]:.4f} | "
            f"val_f1={lc_val_f1[-1]:.4f}"
        )

    fig_lc, ax = plt.subplots(figsize=(8, 5))
    ax.plot(lc_train_sizes, lc_train_f1, marker='o', label='Train F1 (macro)', linewidth=2)
    ax.plot(lc_train_sizes, lc_val_f1, marker='s', label='Validation F1 (macro)', linewidth=2)
    ax.set_title('Learning Curve - DistilBERT', fontweight='bold', fontsize=13)
    ax.set_xlabel('Training set size')
    ax.set_ylabel('Macro F1')
    ax.legend(fontsize=11)
    ax.grid(alpha=0.2)
    plt.tight_layout()
    fig_lc.savefig(f'{output_dir}/03_learning_curve.png', dpi=300, bbox_inches='tight')
    plt.close()

else:
    train_sizes, train_scores, val_scores = learning_curve(
        estimator=learning_model,
        X=X_trainval_tfidf,
        y=y_trainval,
        train_sizes=np.linspace(0.2, 1.0, 5),
        cv=3,
        scoring='f1_macro',
        n_jobs=-1
    )

    fig_lc, ax = plt.subplots(figsize=(8, 5))
    ax.plot(train_sizes, train_scores.mean(axis=1), marker='o', label='Train F1 (macro)', linewidth=2)
    ax.plot(train_sizes, val_scores.mean(axis=1), marker='s', label='Validation F1 (macro)', linewidth=2)
    ax.fill_between(
        train_sizes,
        val_scores.mean(axis=1) - val_scores.std(axis=1),
        val_scores.mean(axis=1) + val_scores.std(axis=1),
        alpha=0.2
    )
    ax.set_title(f'Learning Curve - {learning_model_name}', fontweight='bold', fontsize=13)
    ax.set_xlabel('Training set size')
    ax.set_ylabel('Macro F1')
    ax.legend(fontsize=11)
    ax.grid(alpha=0.2)
    plt.tight_layout()
    fig_lc.savefig(f'{output_dir}/03_learning_curve.png', dpi=300, bbox_inches='tight')
    plt.close()

# Diagramme: Feature importance
feature_names = np.array(tfidf.get_feature_names_out())

feature_model_name = best_model_name
feature_model = best_model
if best_model_name == 'DistilBERT':
    fallback = test_df_metrics[test_df_metrics['Model'] != 'DistilBERT']
    if not fallback.empty:
        feature_model_name = fallback.sort_values('Test F1 (macro)', ascending=False).iloc[0]['Model']
        feature_model = best_models[feature_model_name]
        print(f"Feature-importance fallback (TF-IDF model): {feature_model_name}")

if hasattr(feature_model, 'coef_'):
    coef = feature_model.coef_
    importance = np.mean(np.abs(coef), axis=0)
    top_idx = np.argsort(importance)[-20:]
    top_values = importance[top_idx]
    top_labels = feature_names[top_idx]
    title = f'Top 20 Influential Features - {feature_model_name}'
    fig_fi, ax = plt.subplots(figsize=(9, 7))
    ax.barh(top_labels, top_values, color='#3498db', edgecolor='black')
    ax.set_title(title, fontweight='bold')
    ax.set_xlabel('Importance score')
    plt.tight_layout()
    fig_fi.savefig(f'{output_dir}/04_feature_importance.png', dpi=300, bbox_inches='tight')
    plt.close()
elif hasattr(feature_model, 'feature_importances_'):
    importance = feature_model.feature_importances_
    top_idx = np.argsort(importance)[-20:]
    top_values = importance[top_idx]
    top_labels = feature_names[top_idx]
    title = f'Top 20 Feature Importances - {feature_model_name}'
    fig_fi, ax = plt.subplots(figsize=(9, 7))
    ax.barh(top_labels, top_values, color='#3498db', edgecolor='black')
    ax.set_title(title, fontweight='bold')
    ax.set_xlabel('Importance score')
    plt.tight_layout()
    fig_fi.savefig(f'{output_dir}/04_feature_importance.png', dpi=300, bbox_inches='tight')
    plt.close()

# 3. Création d'un rapport résumé
report_path = f'{output_dir}/summary_report.txt'
with open(report_path, 'w', encoding='utf-8') as f:
    f.write(f"{'='*80}\n")
    f.write(f"RAPPORT D'ANALYSE - MODELE HATE SPEECH DETECTION\n")
    f.write(f"Généré: {datetime.now().strftime('%Y-%m-%d %H:%M:%S')}\n")
    f.write(f"{'='*80}\n\n")
    
    f.write(f"1. MODELE SELECTIONNE\n")
    f.write(f"{'-'*80}\n")
    f.write(f"Meilleur modèle (validation): {best_model_name}\n")
    f.write(f"Accuracy (test): {test_df_metrics.iloc[0]['Test Accuracy']:.4f}\n")
    f.write(f"Macro-F1 (test): {test_df_metrics.iloc[0]['Test F1 (macro)']:.4f}\n")
    f.write(f"Hyperparamètres: {res_df.iloc[0]['Best Params']}\n\n")
    
    f.write(f"2. RESULTATS PAR MODELE (SET TEST)\n")
    f.write(f"{'-'*80}\n")
    for idx, row in test_df_metrics.iterrows():
        f.write(f"\n{row['Model']}:\n")
        f.write(f"  - Accuracy: {row['Test Accuracy']:.4f}\n")
        f.write(f"  - Precision (macro): {row['Test Precision (macro)']:.4f}\n")
        f.write(f"  - Recall (macro): {row['Test Recall (macro)']:.4f}\n")
        f.write(f"  - F1 (macro): {row['Test F1 (macro)']:.4f}\n")
    
    f.write(f"\n3. VALIDATION CROISEE (5-FOLD)\n")
    f.write(f"{'-'*80}\n")
    for idx, row in cv_df.iterrows():
        source = row.get('CV Source', 'kfold')
        if source == 'validation_proxy':
            f.write(f"{row['Model']}: Acc n/a | F1 proxy(val) {row['CV F1 Mean']:.4f}\n")
        else:
            f.write(f"{row['Model']}: Acc {row['CV Accuracy Mean']:.4f}±{row['CV Accuracy Std']:.4f} | F1 {row['CV F1 Mean']:.4f}±{row['CV F1 Std']:.4f}\n")
    
    f.write(f"\n4. DISTRIBUTON ENSEMBLE DE DONNEES\n")
    f.write(f"{'-'*80}\n")
    f.write(f"Taille train: {len(X_train):,}\n")
    f.write(f"Taille validation: {len(X_val):,}\n")
    f.write(f"Taille test: {len(X_test):,}\n")
    f.write(f"Vocabulaire TF-IDF: {len(tfidf.vocabulary_):,} terms\n\n")
    
    f.write(f"5. ANALYSE DES ERREURS (MODELE FINAL)\n")
    f.write(f"{'-'*80}\n")
    f.write(f"Total test: {len(analysis_df):,}\n")
    f.write(f"Mal classifiés: {len(misclassified):,} ({len(misclassified)/len(analysis_df)*100:.1f}%)\n\n")
    f.write(f"Types d'erreurs les plus courants:\n")
    for (true, pred), count in error_types.head(6).items():
        f.write(f"  {true:20} → {pred:20} : {count} occurrences\n")
    
    f.write(f"\n{'='*80}\n")
    f.write(f"Fichiers générés:\n")
    f.write(f"  - validation_results.csv\n")
    f.write(f"  - test_results.csv\n")
    f.write(f"  - cross_validation_results.csv\n")
    f.write(f"  - 01_model_comparison.png\n")
    f.write(f"  - 02_confusion_matrices.png\n")
    f.write(f"  - 03_learning_curve.png\n")
    f.write(f"  - 04_feature_importance.png\n")
    f.write(f"  - missing_values.png\n")
    f.write(f"  - word_count_boxplot.png\n")
    f.write(f"  - summary_report.txt\n")
    f.write(f"\n{'='*80}\n")

print(f"\nSauvegarde terminée!")
print(f"Dossier de sortie: {os.path.abspath(output_dir)}")
print(f"Fichiers générés: 6 diagrammes + 3 CSV + 1 rapport texte")

Some weights of DistilBertForSequenceClassification were not initialized from the model checkpoint at distilbert-base-uncased and are newly initialized: ['classifier.bias', 'classifier.weight', 'pre_classifier.bias', 'pre_classifier.weight']
You should probably TRAIN this model on a down-stream task to be able to use it for predictions and inference.


Learning curve strategy: progressive DistilBERT re-training on raw text.


Map: 100%|██████████| 3717/3717 [00:00<00:00, 205361.56 examples/s]


Epoch,Training Loss,Validation Loss,F1 Macro,Recall Macro,Hate Recall
1,0.247900,0.427927,0.711961,0.788473,0.584112
2,0.073200,0.584508,0.719775,0.782058,0.556075


Map: 100%|██████████| 3659/3659 [00:00<00:00, 162791.00 examples/s]


...

Map: 100%|██████████| 17506/17506 [00:00<00:00, 195802.86 examples/s]


Epoch,Training Loss,Validation Loss,F1 Macro,Recall Macro,Hate Recall
1,0.108100,0.213942,0.814667,0.912797,0.925234
2,0.017800,0.268139,0.824434,0.897224,0.869159


Map: 100%|██████████| 17346/17346 [00:00<00:00, 179538.97 examples/s]


Map: 100%|██████████| 3717/3717 [00:00<00:00, 167947.47 examples/s]


  size=17346 | train_f1=0.9030 | val_f1=0.8244
Feature-importance fallback (TF-IDF model): Logistic Regression

Sauvegarde terminée!
Dossier de sortie: /workspace/Outputs
Fichiers générés: 6 diagrammes + 3 CSV + 1 rapport texte


In [26]:
# 1. Répartition des classes
fig_class, axes = plt.subplots(1, 2, figsize=(14, 5))
counts = df['class'].value_counts().sort_index()
colors = ['#e74c3c', '#f39c12', '#2ecc71']
label_map = {0: 'Hate Speech', 1: 'Offensive Language', 2: 'Neither'}
bars = axes[0].bar([label_map[i] for i in counts.index], counts.values, color=colors, edgecolor='black')
for bar, val in zip(bars, counts.values):
    axes[0].text(bar.get_x() + bar.get_width()/2, bar.get_height() + 200,
                 f'{val:,}', ha='center', fontsize=12, fontweight='bold')
axes[0].set_title('Class Distribution', fontsize=14, fontweight='bold')
axes[0].set_ylabel('Count')
axes[1].pie(counts.values, labels=[label_map[i] for i in counts.index],
            autopct='%1.1f%%', colors=colors, startangle=90,
            textprops={'fontsize': 12}, wedgeprops={'edgecolor': 'black'})
axes[1].set_title('Class Proportions', fontsize=14, fontweight='bold')
plt.tight_layout()
fig_class.savefig(f'{output_dir}/00_class_distribution.png', dpi=300, bbox_inches='tight')
plt.close()

# 2. Analyse de la longueur des tweets
fig_len, axes = plt.subplots(1, 2, figsize=(14, 5))
for cls, color, label in [(0, '#e74c3c', 'Hate Speech'),
                         (1, '#f39c12', 'Offensive'),
                         (2, '#2ecc71', 'Neither')]:
    subset = df[df['class'] == cls]
    axes[0].hist(subset['tweet_length'], bins=50, alpha=0.6, label=label, color=color)
    axes[1].hist(subset['word_count'], bins=30, alpha=0.6, label=label, color=color)
axes[0].set_title('Tweet Length Distribution', fontweight='bold')
axes[0].set_xlabel('Character count')
axes[0].legend()
axes[1].set_title('Word Count Distribution', fontweight='bold')
axes[1].set_xlabel('Word count')
axes[1].legend()
plt.tight_layout()
fig_len.savefig(f'{output_dir}/00_tweet_length_distribution.png', dpi=300, bbox_inches='tight')
plt.close()

# 3. Correlation heatmap
num_cols = [c for c in ['count', 'hate_speech', 'offensive_language', 'neither', 'tweet_length', 'word_count'] if c in df.columns]
corr_df = df[num_cols + ['class']].copy()
plt.figure(figsize=(9, 6))
sns.heatmap(corr_df.corr(numeric_only=True), annot=True, fmt='.2f', cmap='coolwarm', square=True)
plt.title('Correlation Heatmap (Numeric Features + Class)', fontweight='bold')
plt.tight_layout()
plt.savefig(f'{output_dir}/00_correlation_heatmap.png', dpi=300, bbox_inches='tight')
plt.close()

# 4. Missing values
missing_counts = df.isna().sum().sort_values(ascending=False)
missing_counts = missing_counts[missing_counts > 0]

fig_missing, ax = plt.subplots(figsize=(10, 4))
if missing_counts.empty:
    ax.bar(['no_missing_values'], [0], color='#95a5a6', edgecolor='black')
    ax.set_ylim(0, 1)
    ax.set_ylabel('Missing count')
    ax.set_title('Missing Values (no missing values found)', fontweight='bold')
else:
    ax.bar(missing_counts.index.astype(str), missing_counts.values, color='#e67e22', edgecolor='black')
    ax.set_ylabel('Missing count')
    ax.set_title('Missing Values by Column', fontweight='bold')
    ax.tick_params(axis='x', rotation=45)

plt.tight_layout()
fig_missing.savefig(f'{output_dir}/missing_values.png', dpi=300, bbox_inches='tight')
plt.close()

# 5. Word count boxplot by class
box_df = df[['class', 'word_count']].copy()
box_df['class_name'] = box_df['class'].map(label_map)

fig_box, ax = plt.subplots(figsize=(10, 5))
sns.boxplot(
    data=box_df,
    x='class_name',
    y='word_count',
    order=['Hate Speech', 'Offensive Language', 'Neither'],
    palette=['#e74c3c', '#f39c12', '#2ecc71'],
    ax=ax,
)
ax.set_title('Word Count Boxplot by Class', fontweight='bold')
ax.set_xlabel('Class')
ax.set_ylabel('Word count')
plt.tight_layout()
fig_box.savefig(f'{output_dir}/word_count_boxplot.png', dpi=300, bbox_inches='tight')
plt.close()